# Research Module 3: Machine Learning & Image Processing

**AI-Ready Radiology Curriculum**

In this notebook you will:
1. Build machine learning classifiers on radiology data (logistic regression, decision tree)
2. Load, display, and manipulate images in Python
3. Run a pre-trained neural network (MobileNetV2) to classify photos
4. Use MediaPipe to detect and segment hands in your own photos

---

**Prerequisites:** Research Modules 1 and 2 completed (Python basics, pandas, evaluation metrics).

---

## Setup

Install MediaPipe (everything else is pre-installed on Colab).

In [ ]:
!pip install -q mediapipe

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
import requests
from io import BytesIO
from IPython.display import display

print('All libraries loaded successfully.')

In [ ]:
# ============================================================
# IMPORTANT: Replace YOUR-USERNAME with your GitHub username
# ============================================================
GITHUB_USERNAME = 'YOUR-USERNAME'

BASE_URL = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/Bootcamp-AI-for-Medical-Imaging/main/data'
print(f'Base URL: {BASE_URL}')

---

## Part 1: Introduction to Machine Learning

Machine learning (ML) is a method for building systems that learn patterns from data rather than being explicitly programmed.

There are two main types:
- **Supervised learning** — You provide labeled examples (input + correct answer). The model learns to predict the answer for new inputs. Example: given AI confidence scores and modality, predict whether a radiologist will confirm the finding.
- **Unsupervised learning** — No labels. The model finds structure on its own. Example: grouping similar imaging studies together.

In this module we focus on **supervised classification**: predicting a category (confirmed vs. not confirmed) from input features.

### 1.1 Load and Prepare the Data

We will use the same radiology AI findings dataset from Module 2. ML models need numeric inputs, so we convert categorical columns (like modality) into numbers using **one-hot encoding**.

In [ ]:
df = pd.read_csv(f'{BASE_URL}/radiology_ai_findings.csv')
print(f'Loaded {len(df)} studies')
df.head()

In [ ]:
# Select features and target
# Target: what we want to predict
y = df['radiologist_confirmed'].astype(int)

# Features: what we use to make the prediction
# We use ai_confidence (numeric) and modality (categorical)
features = df[['ai_confidence', 'modality']].copy()

# One-hot encode modality: turns one column into multiple 0/1 columns
features_encoded = pd.get_dummies(features, columns=['modality'], dtype=int)

print(f'Features shape: {features_encoded.shape}')
print(f'Feature columns: {list(features_encoded.columns)}')
features_encoded.head()

### 1.2 Train/Test Split

You never evaluate a model on the same data it learned from — that would be like giving a student the exam answers during study, then testing them with the same questions. The model would appear to perform well but fail on new data.

We split the data:
- **Training set** (80%) — the model learns from this
- **Test set** (20%) — held out to measure real performance

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_encoded, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')
print(f'Features per sample: {X_train.shape[1]}')

### 1.3 Logistic Regression

Logistic regression is one of the simplest classifiers. Despite its name, it is used for **classification** (not regression). It learns a weighted combination of features and outputs a probability between 0 and 1.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train the model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)

# Predict on test set
lr_predictions = lr_model.predict(X_test)

# Evaluate
lr_accuracy = accuracy_score(y_test, lr_predictions)
print(f'Logistic Regression Accuracy: {lr_accuracy:.1%}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, lr_predictions))
print()
print('Classification Report:')
print(classification_report(y_test, lr_predictions, target_names=['Not Confirmed', 'Confirmed']))

### 1.4 Decision Tree

A decision tree learns a series of yes/no questions about the features. It splits the data at each step to separate the classes as cleanly as possible.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train the model
dt_model = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_model.fit(X_train, y_train)

# Predict on test set
dt_predictions = dt_model.predict(X_test)

# Evaluate
dt_accuracy = accuracy_score(y_test, dt_predictions)
print(f'Decision Tree Accuracy: {dt_accuracy:.1%}')
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, dt_predictions))
print()
print('Classification Report:')
print(classification_report(y_test, dt_predictions, target_names=['Not Confirmed', 'Confirmed']))

In [ ]:
# Visualize the decision tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt_model, feature_names=list(features_encoded.columns),
          class_names=['Not Confirmed', 'Confirmed'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree: Predicting Radiologist Confirmation', fontsize=14)
plt.tight_layout()
plt.show()

### 1.5 Compare Models

In [ ]:
print('=== Model Comparison ===')
print(f'Logistic Regression: {lr_accuracy:.1%}')
print(f'Decision Tree:       {dt_accuracy:.1%}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, name, preds in zip(axes,
                           ['Logistic Regression', 'Decision Tree'],
                           [lr_predictions, dt_predictions]):
    cm = confusion_matrix(y_test, preds)
    labels = np.array([[f'TN\n{cm[0,0]}', f'FP\n{cm[0,1]}'],
                       [f'FN\n{cm[1,0]}', f'TP\n{cm[1,1]}']])
    import seaborn as sns
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Not Confirmed', 'Confirmed'],
                yticklabels=['Predicted: No', 'Predicted: Yes'],
                annot_kws={'size': 13, 'fontweight': 'bold'}, ax=ax)
    acc = accuracy_score(y_test, preds)
    ax.set_title(f'{name}\nAccuracy: {acc:.1%}', fontsize=12)

plt.tight_layout()
plt.show()

---

## Part 2: Image Basics in Python

AI in radiology works with images. Before using pre-trained models, you need to understand how computers represent images: as grids of numbers (pixel arrays).

### 2.1 Load and Display an Image

We load a chest X-ray from your GitHub repository using PIL (Python Imaging Library) and matplotlib.

In [ ]:
# Load a sample image from your GitHub repo
img_url = f'https://raw.githubusercontent.com/{GITHUB_USERNAME}/Bootcamp-AI-for-Medical-Imaging/main/data/CXR.jpg'

response = requests.get(img_url)
response.raise_for_status()
sample_img = Image.open(BytesIO(response.content))

print(f'Image size: {sample_img.size} (width x height)')
print(f'Image mode: {sample_img.mode}')

plt.figure(figsize=(6, 6))
plt.imshow(sample_img, cmap='gray')
plt.title('Sample Chest X-Ray')
plt.axis('off')
plt.show()

### 2.2 Understanding Pixel Arrays

Every image is stored as a NumPy array of numbers. Each number represents the brightness of one pixel.

- **Grayscale images** have shape `(height, width)` — one number per pixel (0 = black, 255 = white)
- **Color images** have shape `(height, width, 3)` — three numbers per pixel (Red, Green, Blue)

In [ ]:
# Convert image to numpy array
img_array = np.array(sample_img)

print(f'Array shape: {img_array.shape}')
print(f'Data type:   {img_array.dtype}')
print(f'Min value:   {img_array.min()}')
print(f'Max value:   {img_array.max()}')
print()
print('Top-left 5x5 pixel values:')
print(img_array[:5, :5] if img_array.ndim == 2 else img_array[:5, :5, 0])

### 2.3 Basic Image Operations

Before feeding images to a model, you typically resize them, adjust contrast, or convert to grayscale.

In [ ]:
# Resize
resized = sample_img.resize((224, 224))

# Convert to grayscale
grayscale = sample_img.convert('L')

# Adjust brightness
enhancer = ImageEnhance.Brightness(sample_img)
bright = enhancer.enhance(1.5)

# Adjust contrast
enhancer = ImageEnhance.Contrast(sample_img)
high_contrast = enhancer.enhance(2.0)

# Display all versions
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Resized (224x224)', 'Grayscale', 'Brighter (1.5x)', 'High Contrast (2x)']
images = [resized, grayscale, bright, high_contrast]

for ax, title, img in zip(axes, titles, images):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Original size:  {sample_img.size}')
print(f'Resized:        {resized.size}')
print(f'Grayscale mode: {grayscale.mode}')

---

## Part 3: Pre-Trained Image Classification

Training a neural network from scratch requires millions of images and days of computing time. **Transfer learning** lets you use a model someone else already trained and apply it to your own images.

We will use **MobileNetV2**, a lightweight neural network trained on **ImageNet** (a dataset of 1.2 million images across 1,000 everyday categories). MobileNetV2 was developed by Google and is widely used in mobile and embedded applications.

This is the same basic approach used in radiology AI — except medical models are fine-tuned on X-rays and CT scans instead of everyday photos.

### 3.1 Load MobileNetV2

In [ ]:
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# Load pre-trained model
weights = MobileNet_V2_Weights.IMAGENET1K_V1
model = mobilenet_v2(weights=weights)
model.eval()  # Set to evaluation mode (not training)

# Get the preprocessing pipeline and class labels
preprocess = weights.transforms()
categories = weights.meta['categories']

print(f'Model loaded: MobileNetV2')
print(f'Number of output classes: {len(categories)}')
print(f'First 10 classes: {categories[:10]}')

### 3.2 Classify a Sample Image

Let's download a photo and see what MobileNetV2 thinks it is.

In [ ]:
# Download a sample image (Golden Retriever from Wikimedia Commons)
sample_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/500px-YellowLabradorLooking_new.jpg'
response = requests.get(sample_url)
response.raise_for_status()
photo = Image.open(BytesIO(response.content)).convert('RGB')

plt.figure(figsize=(5, 5))
plt.imshow(photo)
plt.title('Sample Photo')
plt.axis('off')
plt.show()

In [ ]:
# Preprocess and run through model
input_tensor = preprocess(photo).unsqueeze(0)  # Add batch dimension

with torch.no_grad():
    output = model(input_tensor)

# Convert to probabilities
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# Get top 5 predictions
top5_prob, top5_idx = torch.topk(probabilities, 5)

print('=== Top 5 Predictions ===')
for i in range(5):
    class_name = categories[top5_idx[i].item()]
    confidence = top5_prob[i].item()
    print(f'{i+1}. {class_name:30s} {confidence:.1%}')

In [ ]:
# Visualize predictions as a bar chart
names = [categories[idx.item()] for idx in top5_idx]
probs = [p.item() for p in top5_prob]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#0EA5E9' if i == 0 else '#CBD5E1' for i in range(5)]
ax.barh(range(4, -1, -1), probs, color=colors)
ax.set_yticks(range(4, -1, -1))
ax.set_yticklabels(names)
ax.set_xlabel('Confidence')
ax.set_title('MobileNetV2 Top-5 Predictions')
ax.set_xlim(0, 1)

for i, (p, name) in enumerate(zip(probs, names)):
    ax.text(p + 0.01, 4 - i, f'{p:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 3.3 Classify Your Own Photo

Take a photo of something on your desk (a pen, a coffee mug, your phone, a pet) with your phone. Transfer it to your computer and upload it below.

MobileNetV2 knows 1,000 categories — see if it recognizes your object.

In [ ]:
from google.colab import files

print('Upload a photo from your phone or computer:')
uploaded = files.upload()

In [ ]:
# Classify your uploaded photo
filename = list(uploaded.keys())[0]
my_photo = Image.open(filename).convert('RGB')

# Show the photo
plt.figure(figsize=(5, 5))
plt.imshow(my_photo)
plt.title('Your Photo')
plt.axis('off')
plt.show()

# Run classification
input_tensor = preprocess(my_photo).unsqueeze(0)

with torch.no_grad():
    output = model(input_tensor)

probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_idx = torch.topk(probabilities, 5)

print('\n=== Top 5 Predictions for Your Photo ===')
for i in range(5):
    class_name = categories[top5_idx[i].item()]
    confidence = top5_prob[i].item()
    print(f'{i+1}. {class_name:30s} {confidence:.1%}')

**Think about it:** MobileNetV2 was trained on everyday photos. What would happen if you fed it a chest X-ray? Try it — upload the CXR image you loaded in Part 2 and see what it predicts. This is why radiology AI models need to be **fine-tuned** on medical images.

---

## Part 4: Image Segmentation with MediaPipe

**Classification** answers: "What is in this image?"

**Segmentation** answers: "Where is each object in this image?" — it labels every pixel.

In radiology, segmentation is used to outline tumors, measure organ volumes, and isolate regions of interest. Here we use **MediaPipe Hands** (by Google) to detect and segment hands — a lightweight model that runs in real time on phones and laptops.

MediaPipe detects **21 landmark points** on each hand (fingertips, knuckles, wrist). We will use these landmarks to create a segmentation mask.

### 4.1 Detect Hand Landmarks

First, let's run MediaPipe on a sample hand image.

In [ ]:
import mediapipe as mp
import cv2

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

print('MediaPipe Hands loaded.')
print(f'Landmark points per hand: {len(mp_hands.HandLandmark)}')

In [ ]:
# Upload a photo of your hand (palm facing camera, against a plain background)
print('Upload a photo of your hand:')
hand_upload = files.upload()

In [ ]:
# Load and process the hand image
hand_filename = list(hand_upload.keys())[0]
hand_img = cv2.imread(hand_filename)
hand_rgb = cv2.cvtColor(hand_img, cv2.COLOR_BGR2RGB)

# Run hand detection
with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5
) as hands:
    results = hands.process(hand_rgb)

if results.multi_hand_landmarks:
    print(f'Detected {len(results.multi_hand_landmarks)} hand(s)')

    # Draw landmarks on the image
    annotated = hand_rgb.copy()
    for hand_landmarks in results.multi_hand_landmarks:
        mp_drawing.draw_landmarks(
            annotated,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style()
        )

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(hand_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(annotated)
    axes[1].set_title('MediaPipe Hand Landmarks (21 points)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No hands detected. Try a clearer photo with your hand against a plain background.')

### 4.2 Create a Segmentation Mask

Now we use the 21 landmark points to create a **convex hull** — a polygon that wraps around all the points. Every pixel inside the hull is labeled as "hand" and every pixel outside is "background."

This is a simplified version of what clinical segmentation models do for tumors, organs, and other structures.

In [ ]:
if results.multi_hand_landmarks:
    h, w, _ = hand_rgb.shape

    # Get landmark coordinates as pixel positions
    points = []
    for lm in results.multi_hand_landmarks[0].landmark:
        points.append([int(lm.x * w), int(lm.y * h)])
    points = np.array(points)

    # Create convex hull mask
    hull = cv2.convexHull(points)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(mask, hull, 255)

    # Apply mask to isolate the hand
    segmented = hand_rgb.copy()
    segmented[mask == 0] = [240, 240, 240]  # Light gray background

    # Display results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(hand_rgb)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Segmentation Mask')
    axes[1].axis('off')

    axes[2].imshow(segmented)
    axes[2].set_title('Segmented Hand')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # Calculate area
    hand_pixels = np.sum(mask > 0)
    total_pixels = h * w
    print(f'Hand area: {hand_pixels:,} pixels ({hand_pixels / total_pixels:.1%} of image)')
else:
    print('No hand was detected above. Re-upload a clearer photo.')

---

## Your Turn

Complete **all three tasks** below.

### Task 1: Try a Different Classifier

Train a **Random Forest** classifier on the radiology data. Random Forest builds many decision trees and combines their votes.

Use `RandomForestClassifier` from `sklearn.ensemble`. Compare its accuracy to the logistic regression and decision tree from Part 1.

In [ ]:
# ============================================================
# TASK 1: Train a Random Forest and compare accuracy
# Hint: from sklearn.ensemble import RandomForestClassifier
# ============================================================

from sklearn.ensemble import RandomForestClassifier

# Your code here:
# 1. Create the model: rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
# 2. Train it: rf_model.fit(X_train, y_train)
# 3. Predict: rf_predictions = rf_model.predict(X_test)
# 4. Print accuracy: accuracy_score(y_test, rf_predictions)



### Task 2: Classify Three Objects

Take photos of **three different objects** (e.g., a water bottle, a shoe, a houseplant). Upload and classify each one with MobileNetV2. For each, record:
- What the object actually is
- What MobileNetV2 predicted (top-1)
- The confidence score

In [ ]:
# ============================================================
# TASK 2: Upload and classify 3 different photos
# ============================================================

# Upload your 3 photos
print('Upload 3 photos:')
task2_upload = files.upload()

# Classify each one
for fname in task2_upload:
    img = Image.open(fname).convert('RGB')
    inp = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        out = model(inp)

    probs = torch.nn.functional.softmax(out[0], dim=0)
    top_prob, top_idx = torch.topk(probs, 3)

    print(f'\n--- {fname} ---')
    for i in range(3):
        print(f'  {i+1}. {categories[top_idx[i].item()]:30s} {top_prob[i].item():.1%}')

### Task 3: Segment Two Hand Poses

Take two photos of your hand in different poses:
1. Hand **open** (fingers spread)
2. Hand in a **fist**

Run the segmentation pipeline on both. Compare the masks. Which pose produces a larger segmented area?

In [ ]:
# ============================================================
# TASK 3: Segment two hand poses and compare
# ============================================================

print('Upload two hand photos (open hand and fist):')
task3_upload = files.upload()

task3_files = list(task3_upload.keys())

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for row, fname in enumerate(task3_files[:2]):
    img = cv2.imread(fname)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = rgb.shape

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=0.5) as hands_det:
        res = hands_det.process(rgb)

    axes[row][0].imshow(rgb)
    axes[row][0].set_title(f'Original: {fname}')
    axes[row][0].axis('off')

    if res.multi_hand_landmarks:
        pts = []
        for lm in res.multi_hand_landmarks[0].landmark:
            pts.append([int(lm.x * w), int(lm.y * h)])
        pts = np.array(pts)
        hull = cv2.convexHull(pts)
        msk = np.zeros((h, w), dtype=np.uint8)
        cv2.fillConvexPoly(msk, hull, 255)
        seg = rgb.copy()
        seg[msk == 0] = [240, 240, 240]

        axes[row][1].imshow(msk, cmap='gray')
        axes[row][1].set_title('Mask')
        axes[row][1].axis('off')

        axes[row][2].imshow(seg)
        area_pct = np.sum(msk > 0) / (h * w)
        axes[row][2].set_title(f'Segmented ({area_pct:.1%} of image)')
        axes[row][2].axis('off')
    else:
        axes[row][1].text(0.5, 0.5, 'No hand detected', ha='center', va='center')
        axes[row][1].axis('off')
        axes[row][2].axis('off')

plt.tight_layout()
plt.show()

---

## Bonus: Video Segmentation

AI segmentation can run on video — processing each frame independently. Record a short clip (3–5 seconds) of your hand opening and closing, upload it, and watch MediaPipe track your hand frame by frame.

This is optional. If you do not have a video to upload, you can skip to the Completion section.

In [ ]:
# Upload a short video (MP4 or MOV, 3-5 seconds)
print('Upload a short video of your hand (optional):')
try:
    video_upload = files.upload()
    video_file = list(video_upload.keys())[0]
except:
    video_file = None
    print('No video uploaded — skipping bonus section.')

In [ ]:
if video_file:
    cap = cv2.VideoCapture(video_file)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Video: {total_frames} frames at {fps:.0f} FPS ({total_frames/fps:.1f} seconds)')

    # Process every 3rd frame to keep it fast
    annotated_frames = []
    frame_idx = 0

    with mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                        min_detection_confidence=0.5,
                        min_tracking_confidence=0.5) as hands_vid:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_idx % 3 == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = hands_vid.process(rgb)
                if res.multi_hand_landmarks:
                    for hlm in res.multi_hand_landmarks:
                        mp_drawing.draw_landmarks(
                            rgb, hlm, mp_hands.HAND_CONNECTIONS,
                            mp_drawing_styles.get_default_hand_landmarks_style(),
                            mp_drawing_styles.get_default_hand_connections_style()
                        )
                annotated_frames.append(rgb)
            frame_idx += 1

    cap.release()
    print(f'Processed {len(annotated_frames)} frames')

    # Show a grid of sampled frames
    n_show = min(8, len(annotated_frames))
    indices = np.linspace(0, len(annotated_frames) - 1, n_show, dtype=int)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, idx in zip(axes.flat, indices):
        ax.imshow(annotated_frames[idx])
        ax.set_title(f'Frame {idx}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Hand Tracking Across Video Frames', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Save as GIF
    from PIL import Image as PILImage
    pil_frames = [PILImage.fromarray(f) for f in annotated_frames]
    pil_frames[0].save('hand_tracking.gif', save_all=True,
                       append_images=pil_frames[1:], duration=100, loop=0)
    print('Saved hand_tracking.gif')
else:
    print('Skipped — no video uploaded.')

---

## Save Your Work

Run the completion record cell below, then save your notebook to GitHub.

### How to save:
1. In Colab, go to **File > Download > Download .ipynb**
2. Go to your forked repository: `https://github.com/YOUR-USERNAME/Bootcamp-AI-for-Medical-Imaging`
3. Click the **Code** tab, then **Add file > Upload files**
4. Drag and drop your downloaded `.ipynb` file
5. Type the commit message: `Completed Research Module 3`
6. Make sure **"Commit directly to the main branch"** is selected
7. Click **Commit changes**

In [ ]:
from datetime import datetime

print('=' * 50)
print('RESEARCH MODULE 3 — COMPLETION RECORD')
print('=' * 50)
print(f'GitHub Username:        {GITHUB_USERNAME}')
print(f'Completed:              {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Logistic Reg Accuracy:  {lr_accuracy:.1%}')
print(f'Decision Tree Accuracy: {dt_accuracy:.1%}')
print(f'MobileNetV2 classes:    {len(categories)}')
print(f'MediaPipe landmarks:    {len(mp_hands.HandLandmark)}')
print('=' * 50)
print('Save this notebook to GitHub to submit your work.')